# Outlier removal, then correlation check, feature removal
## using log t1/2
2025-11-05, Alexander Minidis

2025-11-06 added Support vector with optimization, looks good. 

### Air data, only RDKIT descriptors 11/10

In [ ]:
import sys
from pathlib import Path

notebookdir = Path.cwd().parents[2]
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders
from typing import Any
import pandas as pd
import numpy as np

import sqlalchemy as sa
from sqlalchemy.orm import sessionmaker
from src.db_schema import *
from src.db_utils import get_selected_data
from src.rdkit_tools import MACCS_NAMES
from src.ml_tools import decorrelate, drop_irrelevant_columns

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, explained_variance_score, mean_absolute_error, root_mean_squared_error

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)
%matplotlib inline

In [ ]:
# set directories and filenames, load database
working_dir = Path.cwd().parent.parent
data_dir = working_dir / "processed_data"
database_file = data_dir / "hsbd_t_half_all.db"
engine = sa.create_engine(f"sqlite:///{database_file}")
Session = sessionmaker(bind=engine)

In [ ]:
# get all data
air_data = get_selected_data("air", Session)
soil_data = get_selected_data("soil", Session)
water_data = get_selected_data("water", Session)
sediment_data = get_selected_data("sediment", Session)

air_data = drop_irrelevant_columns(air_data, to_drop=["maccs"])
# water_data = drop_irrelevant_columns(water_data)
# soil_data = drop_irrelevant_columns(soil_data)
# sediment_data = drop_irrelevant_columns(sediment_data)

target_column = "T_half_days"

data_to_use = air_data.copy()

## Preprocessing

In [ ]:
# we will only use air data
X, y = data_to_use.drop(columns=[target_column]), data_to_use[target_column]
print(f"Number of features: {X.shape[1]}, number of samples: {X.shape[0]}")

### 1. Outlier detection

In [ ]:
# Outlier detection using IQR and Z-score methods
# added for poc, not used, using sklearn instead
def detect_outliers_iqr(df, factor=1.5) -> dict:
    outlier_indices = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - factor * IQR
        upper_bound = Q3 + factor * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].index.tolist()
        if outliers:
            outlier_indices[col] = outliers
    return outlier_indices


def detect_outliers_zscore(df, threshold=3) -> dict:
    outlier_indices = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        z_scores = np.abs((df[col] - df[col].mean()) / df[col].std())
        outliers = df[z_scores > threshold].index.tolist()
        if outliers:
            outlier_indices[col] = outliers
    return outlier_indices


# Example usage:
iqr_outliers = detect_outliers_iqr(X)
zscore_outliers = detect_outliers_zscore(X)
iqr_df = pd.DataFrame.from_dict(iqr_outliers, orient="index").reset_index()
zscore_df = pd.DataFrame.from_dict(zscore_outliers, orient="index").reset_index()
print("IQR outliers DataFrame: Outliers are values outside [Q1 - 1.5 x IQR, Q3 + 1.5 x IQR]")
# print(iqr_df.head())
print("Z-score outliers DataFrame: Outliers are values with Z-score > 3")
# print(zscore_df.head())

# print('IQR outliers per feature:', {k: len(v) for k, v in iqr_outliers.items()})
# print('Z-score outliers per feature:', {k: len(v) for k, v in zscore_outliers.items()})

In [ ]:
# Automated outlier detection using sklearn's IsolationForest and LocalOutlierFactor
# IsolationForest
iso = IsolationForest(contamination=0.05, random_state=42)
outlier_pred_iso = iso.fit_predict(X)
iso_outliers = np.where(outlier_pred_iso == -1)[0]
print(f"IsolationForest detected {len(iso_outliers)} outliers.")

# LocalOutlierFactor
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
outlier_pred_lof = lof.fit_predict(X)
lof_outliers = np.where(outlier_pred_lof == -1)[0]
print(f"LocalOutlierFactor detected {len(lof_outliers)} outliers.")

# Optionally, visualize outliers for the first two features
# plt.figure(figsize=(8,6))
# plt.scatter(X.iloc[:,0], X.iloc[:,1], c='blue', label='Inliers')
# plt.scatter(X.iloc[iso_outliers,0], X.iloc[iso_outliers,1], c='red', label='IsolationForest Outliers')
# plt.xlabel(X.columns[0])
# plt.ylabel(X.columns[1])
# plt.legend()
# plt.title('IsolationForest Outlier Detection')
# plt.show()

In [ ]:
# List outliers for each method and label those found in both, including original info columns
# Reload air_data with original columns if needed
air_data_full = get_selected_data("air", Session)
cols_to_add = ["id", "Canonical_smiles", "reference", target_column]
iso_set = set(iso_outliers)
lof_set = set(lof_outliers)
common_outliers = iso_set & lof_set
iso_df = X.iloc[list(iso_set)].copy()
iso_df["outlier_method"] = ["both" if idx in common_outliers else "IsolationForest" for idx in iso_set]
iso_df = iso_df.merge(air_data_full[cols_to_add], left_index=True, right_index=True, how="left")
lof_df = X.iloc[list(lof_set)].copy()
lof_df["outlier_method"] = ["both" if idx in common_outliers else "LocalOutlierFactor" for idx in lof_set]
lof_df = lof_df.merge(air_data_full[cols_to_add], left_index=True, right_index=True, how="left")
# print('IsolationForest outliers:')
# display(iso_df)
# print('LocalOutlierFactor outliers:')
# display(lof_df)
# lof_df.to_csv(working_dir / 'lof_outliers_maccs_only.csv', index=False)
# iso_df.to_csv(working_dir / 'isolationforest_outliers_maccs_only.csv', index=False)

In [ ]:
# Visualize overlap between IsolationForest and LocalOutlierFactor outliers
import matplotlib_venn as venn
import matplotlib.pyplot as plt

iso_set = set(iso_outliers)
lof_set = set(lof_outliers)
plt.figure(figsize=(6, 6))
venn.venn2([iso_set, lof_set], set_labels=("IsolationForest", "LocalOutlierFactor"))
plt.title("Overlap of Outliers Detected by IsolationForest and LocalOutlierFactor")
plt.show()

In [ ]:
# Remove outlier samples detected by isolation forest
mask_inliers = outlier_pred_iso == 1  # outlier_pred_iso is the result of iso.fit_predict(X)
X_clean = X[mask_inliers].copy()
y_clean = y[mask_inliers].copy()
print(f"Cleaned dataset: {X_clean.shape[0]} samples, {X_clean.shape[1]} features.")

In [ ]:
# # Remove outlier samples detected by above methods
# mask_inliers = outlier_pred_iso == 1
# X_clean = X[mask_inliers].copy()
# y_clean = y[mask_inliers].copy()

# # Drop rows with NaN or infinite values in X_clean or y_clean
# mask_notnull = X_clean.notnull().all(axis=1) & y_clean.notnull()
# mask_finite = np.isfinite(X_clean).all(axis=1) & np.isfinite(y_clean)
# final_mask = mask_notnull & mask_finite

# X_clean = X_clean[final_mask]
# y_clean = y_clean[final_mask]

# print(f"Cleaned dataset: {X_clean.shape[0]} samples, {X_clean.shape[1]} features.")
# print(f"NaN values in X_clean: {X_clean.isna().sum().sum()}, in y_clean: {y_clean.isna().sum()}")
# print(f"Infinite values in X_clean: {np.isinf(X_clean).sum().sum()}, in y_clean: {np.isinf(y_clean).sum()}")

### 2. Scaling/normalization

In [ ]:
del X, y
X = X_clean.copy()
y = y_clean.copy()

In [ ]:
# scaling
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_scaled = X_scaled.reset_index(drop=True)
y = y.reset_index(drop=True)

In [ ]:
# remove zero std columns (no variance)
zero_std_cols = X_scaled.columns[X_scaled.std() == 0]
X_scaled = X_scaled.drop(columns=zero_std_cols)
print(f"Number of features: {X_scaled.shape[1]}, number of samples: {X_scaled.shape[0]}")

In [ ]:
# drop columns hihgly correlated to some others
cols_to_drop = decorrelate(X_scaled, target_column, threshold=0.95)
X_decorrelated = X_scaled.drop(columns=cols_to_drop)
print(f"Number of features: {X_decorrelated.shape[1]}, number of samples: {X_decorrelated.shape[0]}")

In [ ]:
# Ensure X_scaled and y are aligned and have no missing values
mask_notnull = X_scaled.notnull().all(axis=1) & y.notnull()
X_scaled = X_scaled[mask_notnull]
y = y[mask_notnull]
print(f"After scaling: X_scaled shape = {X_scaled.shape}, y shape = {y.shape}")

## Model

In [ ]:
def output_metrics(y_true: Any, y_pred: Any) -> None:
    print(f"R2: {r2_score(y_true, y_pred):.3f}")
    print(f"MAE: {mean_absolute_error(y_true, y_pred):.3f}")
    print(f"MSE: {mean_squared_error(y_true, y_pred):.3f}")
    print(f"RMSE: {root_mean_squared_error(y_true, y_pred):.3f}")
    print(f"Explained Variance: {explained_variance_score(y_true, y_pred):.3f}")

In [ ]:
X = X_decorrelated.copy()
y = np.log10(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## RFR

In [ ]:
# Random Forest Regressor with parameter optimization (fixed 'max_features')
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

rf_param_grid = {
    "n_estimators": [100, 300, 500, 700],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
}

rf = RandomForestRegressor(random_state=42)
rf_grid_search = GridSearchCV(rf, rf_param_grid, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
rf_grid_search.fit(X_train, y_train)

print("Best RF parameters:", rf_grid_search.best_params_)
best_rf = rf_grid_search.best_estimator_

y_pred_rf = best_rf.predict(X_test)
y_test_exp_rf = np.power(10, y_test)
y_pred_exp_rf = np.power(10, y_pred_rf)
output_metrics(y_test_exp_rf, y_pred_exp_rf)

## SVR
looking best so far, at least for air_data

In [ ]:
# Support Vector Regression (SVR) model
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

param_grid = {"C": [0.1, 1, 10, 100], "epsilon": [0.01, 0.1, 0.2, 0.5], "kernel": ["rbf", "linear"], "gamma": ["scale", "auto"]}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
best_svr = grid_search.best_estimator_

In [ ]:
# svr = SVR(kernel='rbf', C=1.0, epsilon=0.2)
svr = best_svr
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)

# Inverse transform predictions and targets
y_test_exp_svr = np.power(10, y_test)
y_pred_exp_svr = np.power(10, y_pred_svr)
print("SVR metrics:")
output_metrics(y_test_exp_svr, y_pred_exp_svr)

In [ ]:
# SVR true vs predicted plot
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test_exp_svr, y_pred_exp_svr)
plt.figure(figsize=(7, 7))
plt.scatter(y_test_exp_svr, y_pred_exp_svr, alpha=0.6)
plt.plot([min(y_test_exp_svr), max(y_test_exp_svr)], [min(y_test_exp_svr), max(y_test_exp_svr)], "r--", label="Ideal")
plt.fill_between(
    [min(y_test_exp_svr), max(y_test_exp_svr)],
    [min(y_test_exp_svr) + mae, max(y_test_exp_svr) + mae],
    [min(y_test_exp_svr) - mae, max(y_test_exp_svr) - mae],
    color="gray",
    alpha=0.2,
    label=f"MAE ±{mae:.2f}",
)
plt.xlabel("True y (days)")
plt.ylabel("Predicted y (days)")
plt.title("SVR: True vs Predicted y with MAE region")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Add shaded region for MAE to SVR true vs predicted plot, limited to x/y <= 100
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test_exp_svr, y_pred_exp_svr)
plt.figure(figsize=(7, 7))
# Only plot points where both true and predicted are <= 100
mask = (y_test_exp_svr <= 100) & (y_pred_exp_svr <= 100)
plt.scatter(y_test_exp_svr[mask], y_pred_exp_svr[mask], alpha=0.6)
plt.plot([0, 100], [0, 100], "r--", label="Ideal")
plt.fill_between([0, 100], [mae, 100 + mae], [-mae, 100 - mae], color="gray", alpha=0.2, label=f"MAE ±{mae:.2f}")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.xlabel("True y (days)")
plt.ylabel("Predicted y (days)")
plt.title("SVR: True vs Predicted y with MAE region (y ≤ 100)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot top ten features by permutation importance for SVR
from sklearn.inspection import permutation_importance

result = permutation_importance(svr, X_test, y_test, n_repeats=10, random_state=42, scoring="neg_mean_squared_error")
importances = result.importances_mean
indices = np.argsort(importances)[::-1][:10]
plt.figure(figsize=(10, 5))
plt.bar(range(10), importances[indices])
plt.xticks(range(10), X_test.columns[indices], rotation=90)
plt.title("Top 10 SVR Feature Importances (Permutation)")
plt.ylabel("Mean Importance")
plt.tight_layout()
plt.show()